In [1]:
# Configuración
import sys
from pathlib import Path

# Hace importable el paquete src/ desde notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F
from src.config import load_config
from src.spark_session import get_spark
from src.io import load_split
from src.preprocessing import build_news, enrich_behaviors, explode_impressions

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]
print("Spark", spark.version, "| salida:", OUT)

Spark 4.0.2 | salida: C:\DevProjects\MMDS\RecSysMIND\data\processed\small


In [2]:
prueba = OUT / "_prueba_parquet"
spark.range(5).write.mode("overwrite").parquet(str(prueba))
print(spark.read.parquet(str(prueba)).count(), "filas leídas -> parquet OK")

5 filas leídas -> parquet OK


In [3]:
news_tr, beh_tr = load_split(spark, cfg, "train")
news_dev, beh_dev = load_split(spark, cfg, "dev")

news = build_news(news_tr, news_dev)
behaviors = enrich_behaviors(beh_tr.unionByName(beh_dev), cfg["validation_date"])
impressions_long = explode_impressions(behaviors)

behaviors.groupBy("split", "periodo").count().orderBy("split", "periodo").show()

+-----+-------+------+
|split|periodo| count|
+-----+-------+------+
|  dev|   test| 73152|
|train|  train|126695|
|train|    val| 30270|
+-----+-------+------+



In [4]:
(news.coalesce(1)
     .write.mode("overwrite").parquet(str(OUT / "news")))

(behaviors.drop("impressions")
          .repartition("periodo")
          .write.mode("overwrite").partitionBy("periodo").parquet(str(OUT / "behaviors")))

(impressions_long
     .repartition("periodo")
     .write.mode("overwrite").partitionBy("periodo").parquet(str(OUT / "impressions_long")))

print("Escritura completa")

Escritura completa


In [ ]:
news_p = spark.read.parquet(str(OUT / "news"))
beh_p = spark.read.parquet(str(OUT / "behaviors"))
imp_p = spark.read.parquet(str(OUT / "impressions_long"))

print("news:", news_p.count(), "(esperado 65.238)")
beh_p.groupBy("periodo").agg(F.count("*").alias("impresiones"),
                             F.sum("n_clicks").alias("clics")).orderBy("periodo").show()
imp_p.groupBy("periodo").agg(F.count("*").alias("filas"),
                             F.sum("label").alias("clics")).orderBy("periodo").show()

# Los clics deben cuadrar entre behaviors e impressions_long
beh_p.printSchema()

import os
tam = sum(f.stat().st_size for f in OUT.rglob("*.parquet")) / 1e6
print(f"Tamaño total en parquet: {tam:.1f} MB")

news: 65238 (esperado 65.238)
+-------+-----------+------+
|periodo|impresiones| clics|
+-------+-----------+------+
|   test|      73152|111383|
|  train|     126695|189519|
|    val|      30270| 46825|
+-------+-----------+------+

+-------+-------+------+
|periodo|  filas| clics|
+-------+-------+------+
|   test|2740998|111383|
|  train|4621015|189519|
|    val|1222429| 46825|
+-------+-------+------+

root
 |-- impression_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- history: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = true)
 |-- hist_len: integer (nullable = true)
 |-- imp_len: integer (nullable = true)
 |-- n_clicks: integer (nullable = true)
 |-- periodo: string (nullable = true)

Tamaño total en parquet: 77.7 MB


In [6]:
spark.stop()